# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Access the dataset metadata
meta = dataset.metadata

print(f"Dataset title: {meta.name}\n")
print("Description:")
print(meta.description)


## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
# List all record sets and their fields, referencing by @id
record_sets = dataset.record_sets
print(f"Number of record sets: {len(record_sets)}\n")
for idx, record_set in enumerate(record_sets):
    print(f"Record Set {idx+1}: @id={record_set['@id']}")
    print(f"  Name: {record_set.get('name','N/A')}")
    print(f"  Description: {record_set.get('description','N/A')}")
    fields = record_set.get('field', [])
    if not isinstance(fields, list):
        fields = [fields]
    print("  Fields and columns (@id):")
    for f in fields:
        if isinstance(f, dict):
            print(f"    Field: {f['@id']}")
        else:
            print(f"    Field: {f}")
    print()


In [ ]:
# Optionally, show a sample record from each RecordSet
for record_set in record_sets:
    record_set_id = record_set['@id']
    print(f"Example record for record set: {record_set_id}")
    try:
        for i, record in enumerate(dataset.records(record_set=record_set_id)):
            print(record)
            if i == 0:
                break
    except Exception as e:
        print(f"  Could not load records (possibly a metadata-only record set or download restriction): {e}\n")


## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Prepare a list of record set @id's
record_set_ids = [r['@id'] for r in record_sets]
dataframes = {}

for record_set_id in record_set_ids:
    try:
        records = list(dataset.records(record_set=record_set_id))
        if records:
            dataframes[record_set_id] = pd.DataFrame(records)
            print(f"Loaded DataFrame for {record_set_id} with {len(records)} rows, columns: {dataframes[record_set_id].columns.tolist()}")
        else:
            print(f"Record set {record_set_id} yielded no records.")
    except Exception as e:
        print(f"Error loading records for {record_set_id}: {e}")

# For further demo, pick the first non-empty DataFrame
main_record_set_id = None
for k,v in dataframes.items():
    if not v.empty:
        main_record_set_id = k
        break

if main_record_set_id:
    print(f"\nSelected main record set for analysis: {main_record_set_id}")
    print("Sample columns:", dataframes[main_record_set_id].columns.tolist())
    display(dataframes[main_record_set_id].head())
else:
    print("No record set with data found.")


## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section should include operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

In [ ]:
df = dataframes[main_record_set_id].copy() if main_record_set_id else pd.DataFrame()

if not df.empty:
    # Try to find a numeric field automatically
    numeric_field = None
    for col in df.columns:
        # Try convert first non-null value to float
        sample = df[col].dropna().values
        if len(sample) and pd.api.types.is_numeric_dtype(df[col]):
            numeric_field = col
            break
        else:
            # Try convert to numeric
            try:
                pd.to_numeric(sample[0])
                numeric_field = col
                df[numeric_field] = pd.to_numeric(df[col], errors='coerce')
                break
            except:
                continue
    if numeric_field:
        print(f"Using numeric field for demo analysis: {numeric_field}")
        # Drop missing values
        df_valid = df[[numeric_field]].dropna()

        # Filter using an example threshold - change as appropriate for the field
        threshold = df_valid[numeric_field].quantile(0.5)
        filtered_df = df_valid[df_valid[numeric_field] > threshold]
        print(f"Filtered records with {numeric_field} > {threshold:.2f}:")
        display(filtered_df.head())

        # Normalize
        filtered_df[f"{numeric_field}_normalized"] = (
            filtered_df[numeric_field] - filtered_df[numeric_field].mean()
        ) / filtered_df[numeric_field].std()
        print(f"Normalized {numeric_field} for filtered records:")
        display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

        # Try grouping by another categorical field
        group_field = None
        for col in df.columns:
            if col != numeric_field and df[col].dtype == object:
                group_field = col
                break
        if group_field:
            grouped_df = filtered_df.join(df[group_field]).groupby(group_field).mean(numeric_only=True)
            print(f"Grouped data by {group_field}:")
            display(grouped_df.head())
    else:
        print("No numeric fields found in main data record set.")
else:
    print("No data available for EDA.")


## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if not df.empty and numeric_field:
    # Histogram
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field].dropna(), kde=True)
    plt.xlabel(numeric_field)
    plt.title(f"Distribution of {numeric_field}")
    plt.show()

    # If group_field exists, boxplot
    if group_field:
        plt.figure(figsize=(10,4))
        sns.boxplot(x=group_field, y=numeric_field, data=df)
        plt.title(f"{numeric_field} by {group_field}")
        plt.xticks(rotation=45)
        plt.show()
else:
    print("No data or numeric field available for plotting.")


## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

--
**Note:** This notebook demonstrates methods for loading and exploring FAIR-structured datasets using the `mlcroissant` library. All references to dataset entities (record sets and fields) adhere to their Croissant `@id` specification for reproducibility and traceability. For deeper project-specific analysis, consult the dataset's documentation and schema for the semantics of each field.